# **EDA Notebook**



---
## 0. Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 20.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [2]:
group_name = "group 24"
student_name = "Shameel Zeshan Khader Sheriff"
student_id = "26030371"

In [3]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [4]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [6]:
# No aditional package required

### 0.b Import Packages

In [7]:
import pandas as pd
import altair as alt

---
## B. Data Understanding

In [8]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "special_offer_product.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [9]:
df.head()

,special_offer_id,product_id
0,a1444fcd-4b51-4227-94cf-b13c1a2a6281,8ea7e0d2-fd39-4625-9a5a-e5575f7cfcca
1,a1444fcd-4b51-4227-94cf-b13c1a2a6281,d39463ae-af40-4f64-8583-fa86deec88a9
2,a1444fcd-4b51-4227-94cf-b13c1a2a6281,05385566-38df-4855-94a8-786cae6ffe49
3,a1444fcd-4b51-4227-94cf-b13c1a2a6281,f33d8a41-b92f-4f2a-8f75-76ea2f25722b
4,a1444fcd-4b51-4227-94cf-b13c1a2a6281,26c6c52f-19c1-4760-a72d-a452f0502a1f


In [10]:
df.shape

(538, 2)

In [11]:
df.size

1076

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 538 entries, 0 to 537
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   special_offer_id  538 non-null    object
 1   product_id        538 non-null    object
dtypes: object(2)
memory usage: 8.5+ KB


In [13]:
df.describe(include="all")

,special_offer_id,product_id
count,538,538
unique,15,295
top,a1444fcd-4b51-4227-94cf-b13c1a2a6281,34ad3302-abb1-4d4d-9f75-b1c0c5919313
freq,295,5


In [14]:
df.isnull().sum().sort_values(ascending=False)

,0
special_offer_id,0
product_id,0


In [15]:
df.nunique().sort_values(ascending=False)

,0
product_id,295
special_offer_id,15


In [16]:
df.duplicated().sum()

np.int64(0)

In [46]:
dataset_insights = """
The special_offer_product dataset contains 538 rows and 2 columns.
Each row links a special offer to a product, so this dataset acts as a bridge table between special_offer.csv and product.csv.

The two columns are special_offer_id and product_id, both stored as object/string identifiers.
There are 15 unique special offers and 295 unique products represented in this table.
The most frequent special_offer_id appears 295 times, which suggests that one offer is linked to a large number of products. This is likely the No Discount offer, which may be used as a default offer relationship.

The dataset has no missing values and no duplicate rows, so it is clean from a basic data quality perspective.
Because this dataset only contains identifiers, it does not directly provide numerical modelling features by itself.
However, it is important for data preparation because it allows promotion information from special_offer.csv to be connected to product-level sales and promotion features.

For clustering, this table can be used to create useful promotion exposure features, such as number of offers linked to each product, maximum discount available for each product, and whether a product is associated with discount-based promotions."""

In [18]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `special_offer_id`

In [19]:
# Explore feature of interest: special_offer_id
feature_name = "special_offer_id"

df[feature_name].value_counts().head(10)

,count
special_offer_id,
a1444fcd-4b51-4227-94cf-b13c1a2a6281,295
df7c9acd-bc0c-4ab0-93ec-064e2b93af0d,111
da53c22d-5867-4110-b1d0-8d84bea95ab5,55
aec729b3-e92e-4f2c-971c-4c5d426a99b9,17
c6fdf038-0e61-436b-a074-ba205a1221cc,12
c319131d-f166-40fe-ab38-cc2cb9f370d8,10
c56f47a7-91e8-42c0-ba34-1048ceb3deb0,8
26e5fe80-b12f-43fe-aeb0-f8bd97970c49,7
57594a82-1a1c-497b-98dd-90e46a1c045c,7


In [20]:
# Number of products linked to each special offer
offer_product_counts = (
    df.groupby("special_offer_id")
    .agg(product_count=("product_id", "nunique"))
    .reset_index()
    .sort_values(by="product_count", ascending=False)
)

offer_product_counts.head(10)

,special_offer_id,product_count
7,a1444fcd-4b51-4227-94cf-b13c1a2a6281,295
14,df7c9acd-bc0c-4ab0-93ec-064e2b93af0d,111
13,da53c22d-5867-4110-b1d0-8d84bea95ab5,55
9,aec729b3-e92e-4f2c-971c-4c5d426a99b9,17
12,c6fdf038-0e61-436b-a074-ba205a1221cc,12
10,c319131d-f166-40fe-ab38-cc2cb9f370d8,10
11,c56f47a7-91e8-42c0-ba34-1048ceb3deb0,8
0,26e5fe80-b12f-43fe-aeb0-f8bd97970c49,7
2,57594a82-1a1c-497b-98dd-90e46a1c045c,7
3,6468cb43-e8b4-475f-b705-f4d37f1c59eb,4


In [21]:
alt.Chart(offer_product_counts).mark_bar().encode(
    x=alt.X("special_offer_id:N", title="Special Offer ID", sort="-y"),
    y=alt.Y("product_count:Q", title="Number of Linked Products"),
    tooltip=["special_offer_id", "product_count"]
).properties(
    title="Number of Products Linked to Each Special Offer",
    width=650,
    height=300
)

alt.Chart(...)

In [22]:
feature_1_insights = """
The first feature explored is special_offer_id, which identifies the special offer linked to each product.
Although this column is an identifier, its frequency is useful because it shows how widely each offer is applied across products.

The most frequent special_offer_id appears 295 times, meaning it is linked to 295 unique products.
This is much higher than the other offers, where the next most common offer is linked to 111 products and the third is linked to 55 products.
This suggests that one offer is used as a broad/default offer across many products, while other offers are more targeted.

There are 15 unique special offers in this bridge table and no missing values.
For clustering, the frequency of special offers is useful because it can help create product-level promotion exposure features, such as the number of offers linked to each product or whether a product is linked to broad default offers versus targeted promotional offers.
"""

In [23]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `product_id`

In [24]:
# Explore feature of interest: product_id
feature_name = "product_id"

df[feature_name].value_counts().head(10)

,count
product_id,
34ad3302-abb1-4d4d-9f75-b1c0c5919313,5
f33d8a41-b92f-4f2a-8f75-76ea2f25722b,5
2c372316-0257-442e-ac56-9e7a2d7c1793,5
aa496d61-efa1-44b6-8b6f-a05b18c3ec25,5
05385566-38df-4855-94a8-786cae6ffe49,5
26c6c52f-19c1-4760-a72d-a452f0502a1f,4
36847649-a79a-4f70-9b5d-5c403627ab75,4
997a4288-fa52-4757-83f0-0282eff84b00,4
6ffc6f55-4c9b-4754-9587-48ea030d529d,4


In [25]:
# Number of special offers linked to each product
product_offer_counts = (
    df.groupby("product_id")
    .agg(offer_count=("special_offer_id", "nunique"))
    .reset_index()
    .sort_values(by="offer_count", ascending=False)
)

product_offer_counts.head(15)

,product_id,offer_count
13,05385566-38df-4855-94a8-786cae6ffe49,5
61,34ad3302-abb1-4d4d-9f75-b1c0c5919313,5
52,2c372316-0257-442e-ac56-9e7a2d7c1793,5
280,f33d8a41-b92f-4f2a-8f75-76ea2f25722b,5
211,aa496d61-efa1-44b6-8b6f-a05b18c3ec25,5
293,ff814bac-d3da-4f66-b911-96ae92edb41a,4
25,0fcd0d2f-4f37-40ed-9d9e-c915482f002b,4
16,07f93ab3-8ef3-4951-ad0a-7736bfe4526f,4
253,d5a1745e-512a-4b41-95ef-4cc650f12ee6,4
89,4782606a-0c9b-477e-9e0f-e996d5c298e2,4


In [26]:
alt.Chart(product_offer_counts.head(15)).mark_bar().encode(
    y=alt.Y("product_id:N", title="Product ID", sort="-x"),
    x=alt.X("offer_count:Q", title="Number of Linked Offers"),
    tooltip=["product_id", "offer_count"]
).properties(
    title="Top 15 Products by Number of Linked Special Offers",
    width=600,
    height=350
)

alt.Chart(...)

In [27]:
feature_2_insights = """
The second feature explored is product_id, which identifies the products connected to special offers.
Although product_id is an identifier, analysing its frequency helps show how many offers each product is linked to.

The results show that some products are linked to as many as 5 special offers, while several others are linked to 4 offers.
This suggests that promotion exposure is not evenly distributed across products. Some products are associated with multiple promotional strategies, while others are linked to fewer offers.

There are 295 unique products in this dataset and no missing product_id values.
The chart highlights the top 15 products with the highest number of linked offers.
For clustering, this relationship is useful because the number of offers linked to a product can later be used as a product-level feature representing promotion intensity or promotional exposure.
"""

In [28]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `offer_count`

In [29]:
# Explore relationship feature: number of offers per product
offer_count_distribution = (
    product_offer_counts["offer_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)

offer_count_distribution.columns = ["offer_count", "product_count"]
offer_count_distribution

,offer_count,product_count
0,1,148
1,2,79
2,3,45
3,4,18
4,5,5


In [30]:
# Summary statistics for number of offers per product
product_offer_counts["offer_count"].describe()

,offer_count
count,295.000000
mean,1.823729
std,1.011556
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,5.000000


In [31]:
alt.Chart(offer_count_distribution).mark_bar().encode(
    x=alt.X("offer_count:O", title="Number of Offers Linked to Product"),
    y=alt.Y("product_count:Q", title="Number of Products"),
    tooltip=["offer_count", "product_count"]
).properties(
    title="Distribution of Offer Count per Product",
    width=450,
    height=300
)

alt.Chart(...)

In [32]:
feature_3_insights = """
The third feature explored is the number of special offers linked to each product.
This relationship-level summary is useful because it shows how promotion exposure varies across products.

Among the 295 unique products in this dataset, 148 products are linked to only 1 offer, 79 products are linked to 2 offers, 45 products are linked to 3 offers, 18 products are linked to 4 offers, and 5 products are linked to 5 offers.
The average number of offers per product is approximately 1.82, with a maximum of 5 offers.
This shows that most products have limited promotion exposure, while a smaller group of products are connected to several special offers.

There are no missing values in the product-offer relationships.
For clustering, this summary can later support engineered features such as offer_count, promotion_intensity, and maximum discount exposure per product.
Products with more linked offers may behave differently from products with only one offer, especially when analysing promotion sensitivity, pricing strategy, and product demand.
"""

In [33]:
# Do not modify this code
print_tile(size="h3", key='feature_3_insights', value=feature_3_insights)

### Integration Analysis: Enriching Offer-Product Relationships

In [34]:
# Load related tables for contextual bridge-table analysis
try:
    special_offer_df = pd.read_csv(at.folder_path / "special_offer.csv")
    product_df = pd.read_csv(at.folder_path / "product.csv")
except Exception as e:
    print(e)

special_offer_df.head(), product_df.head()

(                       special_offer_id  min_quantity  max_quantity  \
 0  a1444fcd-4b51-4227-94cf-b13c1a2a6281           0.0           NaN   
 1  df7c9acd-bc0c-4ab0-93ec-064e2b93af0d          11.0          14.0   
 2  da53c22d-5867-4110-b1d0-8d84bea95ab5          15.0          24.0   
 3  aec729b3-e92e-4f2c-971c-4c5d426a99b9          25.0          40.0   
 4  41f713bd-c29a-4ab1-ae06-dbb51ec03ec0          41.0          60.0   
 
                 description  discount_pct             type     category  \
 0               No Discount          0.00      No Discount  No Discount   
 1  Volume Discount 11 to 14          0.02  Volume Discount     Reseller   
 2  Volume Discount 15 to 24          0.05  Volume Discount     Reseller   
 3  Volume Discount 25 to 40          0.10  Volume Discount     Reseller   
 4  Volume Discount 41 to 60          0.15  Volume Discount     Reseller   
 
             start_date             end_date  
 0  2011-05-01 00:00:00  2014-11-30 00:00:00  
 1  2011-05-31

In [35]:
# Check the grain and join keys before integration
print("special_offer_product rows:", len(df))
print("Unique special_offer_id-product_id pairs:", df.drop_duplicates(["special_offer_id", "product_id"]).shape[0])
print("Duplicate bridge rows:", df.duplicated(["special_offer_id", "product_id"]).sum())
print("Unique special_offer_id in bridge:", df["special_offer_id"].nunique())
print("Unique product_id in bridge:", df["product_id"].nunique())

print("\nspecial_offer rows:", len(special_offer_df))
print("Unique special_offer_id in special_offer:", special_offer_df["special_offer_id"].nunique())
print("special_offer_id is unique in special_offer:", special_offer_df["special_offer_id"].is_unique)

print("\nproduct rows before removing exact duplicates:", len(product_df))
print("Unique product_id in product:", product_df["product_id"].nunique())
print("Exact duplicate product rows:", product_df.duplicated().sum())

special_offer_product rows: 538
Unique special_offer_id-product_id pairs: 538
Duplicate bridge rows: 0
Unique special_offer_id in bridge: 15
Unique product_id in bridge: 295

special_offer rows: 16
Unique special_offer_id in special_offer: 16
special_offer_id is unique in special_offer: True

product rows before removing exact duplicates: 886
Unique product_id in product: 504
Exact duplicate product rows: 382


In [36]:
# Remove exact duplicate product rows before joining to the bridge table
product_unique_df = product_df.drop_duplicates().copy()

print("product rows after removing exact duplicates:", len(product_unique_df))
print("Unique product_id after duplicate removal:", product_unique_df["product_id"].nunique())
print("product_id is unique after duplicate removal:", product_unique_df["product_id"].is_unique)

product rows after removing exact duplicates: 504
Unique product_id after duplicate removal: 504
product_id is unique after duplicate removal: True


In [37]:
# Enrich the bridge table with offer and product context
offer_product_enriched_df = df.merge(
    special_offer_df[
        [
            "special_offer_id",
            "description",
            "type",
            "category",
            "discount_pct"
        ]
    ],
    on="special_offer_id",
    how="left",
    validate="many_to_one",
    indicator="offer_join_status"
).merge(
    product_unique_df[
        [
            "product_id",
            "name",
            "product_number",
            "is_sellable",
            "list_price",
            "standard_cost"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="product_join_status"
)

offer_product_enriched_df.head()

,special_offer_id,product_id,description,type,category,discount_pct,offer_join_status,name,product_number,is_sellable,list_price,standard_cost,product_join_status
0,a1444fcd-4b51-4227-94cf-b13c1a2a6281,8ea7e0d2-fd39-4625-9a5a-e5575f7cfcca,No Discount,No Discount,No Discount,0.0,both,"HL Road Frame - Black, 58",FR-R92B-58,1.0,1431.50,1059.3100,both
1,a1444fcd-4b51-4227-94cf-b13c1a2a6281,d39463ae-af40-4f64-8583-fa86deec88a9,No Discount,No Discount,No Discount,0.0,both,"HL Road Frame - Red, 58",FR-R92R-58,1.0,1431.50,NaN,both
2,a1444fcd-4b51-4227-94cf-b13c1a2a6281,05385566-38df-4855-94a8-786cae6ffe49,No Discount,No Discount,No Discount,0.0,both,"Sport-100 Helmet, Red",HL-U509-R,1.0,34.99,13.0863,both
3,a1444fcd-4b51-4227-94cf-b13c1a2a6281,f33d8a41-b92f-4f2a-8f75-76ea2f25722b,No Discount,No Discount,No Discount,0.0,both,"Sport-100 Helmet, Black",HL-U509,1.0,34.99,13.0863,both
4,a1444fcd-4b51-4227-94cf-b13c1a2a6281,26c6c52f-19c1-4760-a72d-a452f0502a1f,No Discount,No Discount,No Discount,0.0,both,"Mountain Bike Socks, M",SO-B909-M,1.0,9.50,3.3963,both


In [38]:
# Check join coverage after enrichment
offer_join_coverage = (
    offer_product_enriched_df["offer_join_status"]
    .value_counts(dropna=False)
    .rename_axis("offer_join_status")
    .reset_index(name="bridge_rows")
)

offer_join_coverage["percentage"] = (
    offer_join_coverage["bridge_rows"] / len(offer_product_enriched_df) * 100
).round(2)

offer_join_coverage

,offer_join_status,bridge_rows,percentage
0,both,538,100.0
1,left_only,0,0.0
2,right_only,0,0.0


In [39]:
product_join_coverage = (
    offer_product_enriched_df["product_join_status"]
    .value_counts(dropna=False)
    .rename_axis("product_join_status")
    .reset_index(name="bridge_rows")
)

product_join_coverage["percentage"] = (
    product_join_coverage["bridge_rows"] / len(offer_product_enriched_df) * 100
).round(2)

product_join_coverage

,product_join_status,bridge_rows,percentage
0,both,538,100.0
1,left_only,0,0.0
2,right_only,0,0.0


In [40]:
offer_product_coverage_summary = (
    offer_product_enriched_df
    .groupby(
        ["description", "type", "category", "discount_pct"],
        as_index=False
    )
    .agg(
        linked_product_count=("product_id", "nunique")
    )
    .sort_values("linked_product_count", ascending=False)
)

offer_product_coverage_summary

,description,type,category,discount_pct,linked_product_count
5,No Discount,No Discount,No Discount,0.00,295
11,Volume Discount 11 to 14,Volume Discount,Reseller,0.02,111
12,Volume Discount 15 to 24,Volume Discount,Reseller,0.05,55
13,Volume Discount 25 to 40,Volume Discount,Reseller,0.10,17
1,LL Road Frame Sale,Excess Inventory,Reseller,0.35,12
10,Touring-3000 Promotion,New Product,Reseller,0.15,10
3,Mountain-100 Clearance Sale,Discontinued Product,Reseller,0.35,8
0,Half-Price Pedal Sale,Seasonal Discount,Customer,0.50,7
4,Mountain-500 Silver Clearance Sale,Discontinued Product,Reseller,0.40,7
9,Touring-1000 Promotion,New Product,Reseller,0.20,4


In [41]:
alt.Chart(offer_product_coverage_summary).mark_bar().encode(
    x=alt.X("description:N", title="Special Offer", sort="-y"),
    y=alt.Y("linked_product_count:Q", title="Number of Linked Products"),
    color=alt.Color("type:N", title="Offer Type"),
    tooltip=[
        "description",
        "type",
        "category",
        alt.Tooltip("discount_pct:Q", format=".2f", title="Offer Discount"),
        "linked_product_count"
    ]
).properties(
    title="Product Coverage by Special Offer Rule",
    width=700,
    height=350
)

alt.Chart(...)

In [42]:
# Product-level promotion exposure summary from the offer-product bridge
product_promotion_summary = (
    offer_product_enriched_df
    .groupby("product_id", as_index=False)
    .agg(
        linked_offer_count=("special_offer_id", "nunique"),
        promotional_offer_count=("discount_pct", lambda x: (x > 0).sum()),
        max_offer_discount_pct=("discount_pct", "max"),
        offer_type_count=("type", "nunique")
    )
)

product_promotion_summary.sort_values(
    ["linked_offer_count", "max_offer_discount_pct"],
    ascending=False
).head(15)

,product_id,linked_offer_count,promotional_offer_count,max_offer_discount_pct,offer_type_count
13,05385566-38df-4855-94a8-786cae6ffe49,5,4,0.15,3
52,2c372316-0257-442e-ac56-9e7a2d7c1793,5,4,0.15,2
61,34ad3302-abb1-4d4d-9f75-b1c0c5919313,5,4,0.15,3
211,aa496d61-efa1-44b6-8b6f-a05b18c3ec25,5,4,0.15,2
280,f33d8a41-b92f-4f2a-8f75-76ea2f25722b,5,4,0.15,3
176,8fde1217-d397-49d8-bef4-08eb29a0806b,4,3,0.40,3
283,f76ed752-5f96-47e4-bb09-30523d045528,4,3,0.30,3
122,681d729f-da4f-4ee4-a4bc-cc67e8501332,4,3,0.20,3
253,d5a1745e-512a-4b41-95ef-4cc650f12ee6,4,3,0.20,3
133,6ffc6f55-4c9b-4754-9587-48ea030d529d,4,3,0.15,3


In [43]:
promotion_count_distribution = (
    product_promotion_summary["promotional_offer_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)

promotion_count_distribution.columns = [
    "promotional_offer_count",
    "product_count"
]

alt.Chart(promotion_count_distribution).mark_bar().encode(
    x=alt.X("promotional_offer_count:O", title="Promotional Offers Linked to Product"),
    y=alt.Y("product_count:Q", title="Number of Products"),
    tooltip=["promotional_offer_count", "product_count"]
).properties(
    title="Distribution of Promotional Offer Links per Product",
    width=500,
    height=300
)

alt.Chart(...)

In [44]:
offer_product_integration_insights = """
The special_offer_product dataset is a bridge table rather than a standalone fact or dimension table. Its grain is one row per special_offer_id-product_id relationship. This is confirmed by 538 rows, 538 unique offer-product pairs, and no duplicate bridge relationships.

This table describes product eligibility or linkage to offer rules, while sales_order_detail describes which offer was actually applied when a product was sold on an order line.

To make the bridge table interpretable, it was integrated with special_offer and product. The special_offer join adds offer descriptions, offer type, category, and discount percentage, while the product join adds product context. Before the product join, exact duplicate rows in product were removed because product contained 886 rows but only 504 unique product_id values. After duplicate removal, product_id became unique and the bridge-to-product join could be checked as many relationships to one product record.

Both integrations have complete coverage. All 538 bridge rows match a record in special_offer and all 538 bridge rows match a product record. This confirms that the observed offer-product relationships can be enriched without losing bridge records.

The enriched offer coverage shows that promotion relationships are unevenly distributed. The No Discount rule is linked to all 295 products represented in the bridge table. Among promotional rules, Volume Discount 11 to 14 has the widest product coverage, followed by Volume Discount 15 to 24. Other seasonal, clearance, excess-inventory, and new-product offers are linked to much smaller sets of products.

The product-level aggregation is important for the selected clustering use case. A product can be linked to multiple offers, so the bridge table cannot be joined directly into a one-row-per-product modelling table without creating repeated product rows. Instead, it can be aggregated into product-level promotion features such as linked_offer_count, promotional_offer_count, max_offer_discount_pct, and offer_type_count.

The promotional offer distribution shows that many products have no promotional offer link beyond the No Discount rule, while a smaller set of products is linked to several promotional offers. This makes the bridge table useful for describing promotion exposure in later product segmentation, alongside demand, price, cost, and sales behaviour features.
"""

In [45]:
print_tile(
    size="h3",
    key="offer_product_integration_insights",
    value=offer_product_integration_insights
)